# Compute virus' effect size on gene expression

In [ ]:
import warnings, time, os, json, time, pickle
from typing import Dict, Any, Optional, Tuple, Iterable, Mapping, Callable
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from __future__ import annotations

import utils_effect_size as es
import utils_analyze_effect_size as utils

import importlib
importlib.reload(es)
importlib.reload(utils)


In [ ]:
# output dir
# All paths are resolved relative to this release directory (the folder that contains this
# notebook), so the bundled `data/` directory is used. Launch Jupyter from this folder, or
# set the VIRUS_EXPR_SHIFT_DIR environment variable to point REPO_DIR elsewhere.
REPO_DIR = os.environ.get("VIRUS_EXPR_SHIFT_DIR", os.getcwd())
data_output_dir = os.path.join(REPO_DIR, "data", "output")
if not os.path.exists(data_output_dir):
    os.makedirs(data_output_dir)

In [ ]:
distance = "median_shift"
load_effect_size_from_file = True
load_effect_distro_from_file = True
keep_only_fish_associated_viruses = True
virus_to_remove = ["Severe acute respiratory syndrome coronavirus 2"]

## load data

### load Log2(TMM-CPM + prior_count) table

In [ ]:
# inputs
data_input_dir = os.path.join(REPO_DIR, "data")
parquet_name = "counts_tmm-norm.parquet"
parquet_path = os.path.join(data_input_dir, parquet_name)

In [ ]:
# The full TMM-normalized count matrix (counts_tmm-norm.parquet, ~9.7 GB) is NOT bundled with
# this code release (see README). It is needed only to recompute the effect sizes from
# scratch. When it is absent, the notebook runs in "load precomputed results" mode and
# regenerates the figures from the bundled tables in data/output/ (this is the default; see
# load_effect_size_from_file / load_effect_distro_from_file above).
HAVE_COUNT_MATRIX = os.path.exists(parquet_path)
if HAVE_COUNT_MATRIX:
    t0 = time.time()
    log2tmmcpm = utils.read_parquet(parquet_path, index_col=None)
    t1 = time.time()
    print(f"loaded df in {t1-t0:.2f} seconds")
    print(f"df shape: {log2tmmcpm.shape}")
else:
    log2tmmcpm = None
    print(f"count matrix not found at:\n  {parquet_path}")
    print("=> running in 'load precomputed results' mode; figures will be rebuilt from "
          "data/output/. Keep load_effect_size_from_file = load_effect_distro_from_file = True "
          "(the default) to use this mode.")

### load gene id conversion table

In [ ]:
path_gene_id_conversion = os.path.join(data_input_dir, "drerio_z12-geneid_to_z11-ENSDARG.csv")
gene_id_to_ensembl_id_df = pd.read_csv(path_gene_id_conversion)
ensembl_id_to_gene_id_dict = dict(zip(gene_id_to_ensembl_id_df['Ensembl_gene_id'], gene_id_to_ensembl_id_df['gene_id']))
ensembl_id_to_gene_id_dict["Run"] = "Run" # preserve the run column

## translate the gene ids to ensembl gene ids

In [ ]:
# translate the gene ids to ensembl gene ids
if log2tmmcpm is not None:
    log2tmmcpm.columns = log2tmmcpm.columns.map(ensembl_id_to_gene_id_dict)
    log2tmmcpm = log2tmmcpm.loc[:, log2tmmcpm.columns.notna()]
    print(f"translated the gene ids to ensembl gene ids, shape of counts table: {log2tmmcpm.shape}", flush=True)

## compute virus positive runs

In [ ]:
csv_path = os.path.join(data_input_dir, "all.quantt_sept24_fishassociated_mar26clusters.tsv")
fish_vir_quant_df = pd.read_csv(csv_path, sep="\t")
#set the index to be the run name
fish_vir_quant_df.set_index("run_name", inplace=True)
#remove the _salmon suffix
fish_vir_quant_df.index = fish_vir_quant_df.index.str.removesuffix("_salmon")

# produce species2runs dict
species2runs = es.df_nonzero_rows_by_column_to_json(fish_vir_quant_df, os.path.join(data_input_dir, "preprocessed", "fish_vir_species2runs_mar26.json"))

species2runs_filtered = {}

## recompute virus positive runs with the log2tmmcpm df

In [ ]:
# # load virus positive runs
# scope = "fish"
# species2runs_path = os.path.join(data_input_dir, 'preprocessed', f'{scope}_virus_species2runs_sorted.json')
# with open(species2runs_path, "r", encoding="utf-8") as f:
#     species2runs = json.load(f)
# species2runs_filtered = {}

In [ ]:
# remove runs if it no longer exists in log2tmmcpm
# (only needed when recomputing from the count matrix; in load-from-file mode the virus set
#  is taken directly from the precomputed effect-size table in the next cells.)
if log2tmmcpm is not None:
    filtered_runs = log2tmmcpm.index.to_list()
    t0 = time.time()
    for i,k in enumerate(species2runs):
        v = species2runs[k]
        v = [item.removesuffix("_salmon") for item in v if item.removesuffix("_salmon") in filtered_runs]
        species2runs_filtered[k] = v
    t1 = time.time()
    print(f"filtered runs in {t1-t0:.2f} seconds")

In [ ]:
# check run numbers
selected_virus = []
if log2tmmcpm is not None:
    species2runs_filtered = dict(sorted(species2runs_filtered.items(), key=lambda kv: len(kv[1]), reverse=True))
    for i,k in enumerate(species2runs_filtered):
        if len(species2runs_filtered[k])>10:
            print(f"{k}\t{len(species2runs_filtered[k])}")
            selected_virus.append(k)
else:
    # load-from-file mode: the columns of the precomputed effect-size table are exactly the
    # viruses that passed the ">10 samples" filter when the analysis was originally run.
    selected_virus = list(
        pd.read_csv(os.path.join(data_output_dir, f"{distance}_stat_df.csv"), index_col=0).columns
    )
print(len(selected_virus))

In [ ]:
# keey only fish-associated viruses
if keep_only_fish_associated_viruses:
    selected_virus = [i for i in selected_virus if not "endogenous_or_nonfish" in i.lower()]
    selected_virus = [i for i in selected_virus if not "insufficient_evidence" in i.lower()]
    print(f"number of fish-associated viruses: {len(selected_virus)}")


In [ ]:
# custom removal 
custom_removal_viruses = ["Myotis ricketti associated fish calicivirus"]
selected_virus = [i for i in selected_virus if i not in custom_removal_viruses]
print(f"number of fish-associated viruses after custom removal: {len(selected_virus)}")


## load genes

### load immune genes

In [ ]:
path = os.path.join(data_input_dir, "immune_genes_v20260311.csv")
immune_gene_df = pd.read_csv(path)

In [ ]:
immune_gene_df

## effect size

In [ ]:
gene_list = immune_gene_df["Gene"].to_list()
virus_list = selected_virus
# filter gene_list (only when the count matrix is available; gene_list / virus_list are used
# only on the recompute path in the next cell)
if log2tmmcpm is not None:
    gene_list = [i for i in gene_list if i in log2tmmcpm.columns.to_list()]

In [ ]:
if load_effect_size_from_file:  
    stat_df = pd.read_csv(os.path.join(data_output_dir, f"{distance}_stat_df.csv"), index_col=0)
    ci_df = pd.read_csv(os.path.join(data_output_dir, f"{distance}_ci_df.csv"), index_col=0)
else:
    stat_df, ci_df = utils.compute_pairwise_stat_ci_grid(
        list_a=gene_list,
    list_b=virus_list,
    fetch_vals_fn=utils.get_expr_vals,
    compute_fn=es.compute_median_shift,
    fetch_kwargs={"counts_table": log2tmmcpm, "species2runs": species2runs},
    compute_kwargs={"n_boot": 400, "alpha": 0.05, "seed": 42},
    ci_fmt="[{:0.2f}, {:0.2f}]",
    progress=True,
    )
    stat_df.to_csv(os.path.join(data_output_dir, f"{distance}_stat_df.csv"))
    ci_df.to_csv(os.path.join(data_output_dir, f"{distance}_ci_df.csv"))

## compute shift for *all genes, *for all viruses

In [ ]:
# compute shift for *all genes, *for all viruses
shift_distro_path = os.path.join(data_output_dir, "median_shift_all_gene_all_virus.pkl")
if load_effect_distro_from_file:
    with open(shift_distro_path, "rb") as f:
        all_shift = pickle.load(f)
else:
    all_shift = {}
    for vir in virus_list:
        df = utils.compute_stat_all_genes(
        list_a=log2tmmcpm.columns.to_list(),
        virus_name=vir,
        fetch_vals_fn=utils.get_expr_vals,
        compute_fn=es.compute_median_shift,
        fetch_kwargs={"counts_table": log2tmmcpm, "species2runs": species2runs},
        compute_kwargs={"n_boot": 1, "alpha": 0.05, "seed": 42},
        n_workers = 12,
        chunk_size= 1000,
        )
        all_shift[vir] = df
    with open(shift_distro_path, "wb") as f:
        pickle.dump(all_shift, f, protocol=pickle.HIGHEST_PROTOCOL)

    
    

In [ ]:
all_shift.keys()

## thresholds


In [ ]:
# compute threshold for each column
threshold_by_column = (
    stat_df.columns.to_series()
    .map(lambda col: utils.thresholds_for_column(col_name=col, df=all_shift, method="tukey"))
)

In [ ]:
# selected virus must have at least one immune gene with a shift > Q3 + 1.5*IQR or shift < Q1 - 1.5*IQR
too_small_log2FC = stat_df.columns[
    (stat_df.max() < pd.Series(threshold_by_column).reindex(stat_df.columns).str[0]) &
    (stat_df.min() > pd.Series(threshold_by_column).reindex(stat_df.columns).str[1])
]

In [ ]:
# make a df of stat_df.max() and threshold_by_column
df_max_threshold = pd.DataFrame({
    "virus": stat_df.columns,
    "min immune gene shift": stat_df.min(),
    "max immune gene shift": stat_df.max(),
    "Q1-1.5*IQR": pd.Series(threshold_by_column).reindex(stat_df.columns).str[1],
    "Q3+1.5*IQR": pd.Series(threshold_by_column).reindex(stat_df.columns).str[0],
})
df_max_threshold

In [ ]:
# parse virus category and update all dataframe with virus names
vir_cleaned, vir_labels = utils.parse_virus_categories(stat_df.columns.to_list(), default_label = "fish-associated")
stat_df.columns = vir_cleaned

_cleaned, _labeled = utils.parse_virus_categories(ci_df.columns.to_list(), default_label = "fish-associated")
ci_df.columns = _cleaned

_cleaned, _labeled = utils.parse_virus_categories(df_max_threshold.columns.to_list(), default_label = "fish-associated")
df_max_threshold.columns = _cleaned

_cleaned, _labeled = utils.parse_virus_categories(df_max_threshold.index.to_list(), default_label = "fish-associated")
df_max_threshold.index = _cleaned

_cleaned, _labeled = utils.parse_virus_categories(list(all_shift.keys()), default_label = "fish-associated")
all_shift_name_update = {new_k: all_shift[old_k] for old_k, new_k in zip(all_shift.keys(), _cleaned)}

In [ ]:
# virus removal

_cleaned, _labeled = utils.parse_virus_categories(selected_virus, default_label = "fish-associated")
# remove columns that are not in the selected_virus list
stat_df = stat_df[_cleaned]
ci_df = ci_df[_cleaned]
df_max_threshold = df_max_threshold.loc[_cleaned]
all_shift_name_update = {k: all_shift_name_update[k] for k in _cleaned}

In [ ]:
fig_height = 18
fig_width = 12
col_categories = dict(zip(vir_cleaned, vir_labels))
if keep_only_fish_associated_viruses:
    fig_height = 18
    fig_width = 6
    col_categories = None

# Suppress plt.show() to modify figure before display
_orig_show = plt.show
plt.show = lambda *a, **kw: None

cg = utils.plot_clustered_heatmap_with_ci_and_violin_and_symbols(
    stat_df, ci_df,
    row_categories=dict(zip(immune_gene_df['Gene'], immune_gene_df['Category'])),
    col_categories=col_categories,
    col_nulls=all_shift_name_update,
    annot_fontsize=7,
    figsize=(fig_width, fig_height),
    cmap = "RdBu_r",
    cbar_label="log2 median shift",
    x_label = "Virus",
    y_label = "Gene",
    upper_thresholds=None,#df_max_threshold["Q3+1.5*IQR"],
    lower_thresholds=None,#df_max_threshold["Q1-1.5*IQR"],
    savepath=None,

    # hide ci test and center the triangles
    show_ci_text=False,
    upper_pos=(0.50, 0.45),
    lower_pos=(0.50, 0.55),

    # color strip thickness
    colors_ratio=(0.015, 0.01),

    upper_fontsize=7,
    lower_fontsize=7,

)

plt.show = _orig_show

# Flip column order by inverting x-axis on all column-aligned axes
fig = getattr(cg, 'fig', None) or getattr(cg, 'figure', None)
hm_xlim = cg.ax_heatmap.get_xlim()
for ax in fig.axes:
    if np.allclose(ax.get_xlim(), hm_xlim, atol=0.1):
        ax.invert_xaxis()

# Flip rows vertically (top row -> bottom row) by inverting y-axis on all row-aligned axes
hm_ylim = cg.ax_heatmap.get_ylim()
for ax in fig.axes:
    if np.allclose(ax.get_ylim(), hm_ylim, atol=0.1):
        ax.invert_yaxis()

fig.savefig(
    os.path.join(data_output_dir, f"{distance}_fishOnly={keep_only_fish_associated_viruses}.svg"),
    dpi=300, bbox_inches="tight", pad_inches=0.02, transparent=True,
)
plt.show()

## endogenous / non-fish viruses

In [ ]:
# heatmap for endogenous / non-fish viruses
# (these are excluded from the fish-associated heatmap above when keep_only_fish_associated_viruses=True)

# the in-memory stat_df / ci_df were renamed & filtered to fish-only above, so re-load from the
# saved CSVs, which still contain every virus (including the endogenous_or_nonfish ones).
stat_df_full = pd.read_csv(os.path.join(data_output_dir, f"{distance}_stat_df.csv"), index_col=0)
ci_df_full   = pd.read_csv(os.path.join(data_output_dir, f"{distance}_ci_df.csv"),   index_col=0)

# select the endogenous / non-fish viruses and strip the category prefix for display
endo_keyword = "endogenous_or_nonfish"
endo_virus_raw = [c for c in stat_df_full.columns if endo_keyword in c.lower()]
cleaned_endo, _ = utils.parse_virus_categories(endo_virus_raw, default_label="endogenous_or_nonfish")
raw_to_clean = dict(zip(endo_virus_raw, cleaned_endo))

stat_df_endo   = stat_df_full[endo_virus_raw].rename(columns=raw_to_clean)
ci_df_endo     = ci_df_full[endo_virus_raw].rename(columns=raw_to_clean)
# all_shift keeps the original (prefixed) keys; pair each with its cleaned name for the null violins
col_nulls_endo = {raw_to_clean[r]: all_shift[r] for r in endo_virus_raw if r in all_shift}
print(f"endogenous / non-fish viruses: {len(endo_virus_raw)} (null distributions found for {len(col_nulls_endo)})")

# scale width to the virus count (the fish-only panel used width 6 for 18 viruses)
fig_height = 18
fig_width  = max(4.0, 6.0 * len(endo_virus_raw) / 18.0)

# Suppress plt.show() to modify figure before display
_orig_show = plt.show
plt.show = lambda *a, **kw: None

cg = utils.plot_clustered_heatmap_with_ci_and_violin_and_symbols(
    stat_df_endo, ci_df_endo,
    row_categories=dict(zip(immune_gene_df['Gene'], immune_gene_df['Category'])),
    col_categories=None,
    col_nulls=col_nulls_endo,
    annot_fontsize=7,
    figsize=(fig_width, fig_height),
    cmap = "RdBu_r",
    cbar_label="log2 median shift",
    x_label = "Virus",
    y_label = "Gene",
    upper_thresholds=None,
    lower_thresholds=None,
    savepath=None,

    # hide ci test and center the triangles
    show_ci_text=False,
    upper_pos=(0.50, 0.45),
    lower_pos=(0.50, 0.55),

    # color strip thickness
    colors_ratio=(0.015, 0.01),

    upper_fontsize=7,
    lower_fontsize=7,
)

plt.show = _orig_show

# Flip column order by inverting x-axis on all column-aligned axes
fig = getattr(cg, 'fig', None) or getattr(cg, 'figure', None)
hm_xlim = cg.ax_heatmap.get_xlim()
for ax in fig.axes:
    if np.allclose(ax.get_xlim(), hm_xlim, atol=0.1):
        ax.invert_xaxis()

# Flip rows vertically (top row -> bottom row) by inverting y-axis on all row-aligned axes
hm_ylim = cg.ax_heatmap.get_ylim()
for ax in fig.axes:
    if np.allclose(ax.get_ylim(), hm_ylim, atol=0.1):
        ax.invert_yaxis()

fig.savefig(
    os.path.join(data_output_dir, f"{distance}_endogenous_nonfish.svg"),
    dpi=300, bbox_inches="tight", pad_inches=0.02, transparent=True,
)
plt.show()